# 06 — Stylized Case: 15 Fields, 5 Hosts

Overview of a larger stylized NCS tieback portfolio.
Generates synthetic field and host data, runs multi-objective optimization,
and shows the main results.

**Problem:** Assign 15 oil fields to 5 host platforms, choosing well counts and development timing
to maximize portfolio Real Option Value (ROV) while managing downside risk (CVaR).

---
# Part A: Data Overview

## 1. Setup & Run Optimization

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import copy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.spatial.distance import cdist

from src.orchestrator import run_optimization
from src.optimizer.encoding import decode_particle
from src.data_generation.generator import generate_data
from src.data_generation.profiles import make_oil_profile, logistic_watercut
from src.valuation.capacity import compute_capped_production
from src.valuation.npv import compute_field_npvs
from src.optimizer.evaluation import evaluate_portfolio
from src.price.schwartz_smith import generate_price_scenarios_final
from src.models import PriceScenarios, CostData

logging.basicConfig(level=logging.INFO, format='%(name)s - %(message)s')
print('Imports OK')

In [ ]:
overrides = {
    'n_fields': 15,
    'n_hosts': 5,
    'experiment_name': 'stylized_15f5h',
    'pso.n_particles': 300,
    'pso.max_iterations': 300,
    'pso.archive_size': 150,
    'price.n_scenarios': 100,
    'price.n_scenarios_final': 5000,
    # Lower host capacities so constraints are binding with 15 fields
    # 'data_generation.host_oil_capacity_min': 0.3,
    # 'data_generation.host_oil_capacity_max': 1.0,
    # 'data_generation.host_gas_capacity_min': 0.1,
    # 'data_generation.host_gas_capacity_max': 0.4,
    # 'data_generation.host_water_capacity_min': 0.3,
    # 'data_generation.host_water_capacity_max': 1.5,
    # Place hosts at K-means centroids of field positions
    'data_generation.host_placement': 'clustered',
}

result = run_optimization(config_path='../config/default.yaml', overrides=overrides, verbose=True)
archive = result['archive']
pareto_front = result['pareto_front']
config = result['config']
history = result['history']
print(f'\nArchive size: {len(archive)}, Pareto front: {len(pareto_front)}')

In [ ]:
# Regenerate data (same seed = same data)
field_data, host_data, cost_data = generate_data(config)

n_fields = config['n_fields']
n_hosts = config['n_hosts']
n_years = config['n_years']
max_start_year = config['max_start_year']
max_wells = config['data_generation']['max_wells']
per_well_rate = config['data_generation']['per_well_rate']
val_cfg = config['valuation']
barrel = val_cfg['barrel']
btu = val_cfg['btu']
nok_per_usd = val_cfg['nok_per_usd']
years = np.arange(n_years)

# Distance and CAPEX matrices
dist_matrix = cdist(field_data.coords, host_data.coords)
capex_matrix = cost_data.capex(field_data.n_wells)

# Use pareto_front (non-dominated under re-evaluated objectives)
obj_final = np.array([s['objectives_final'] for s in pareto_front])
best_rov_idx = np.argmax(obj_final[:, 0])
best_cvar_idx = np.argmax(obj_final[:, 1])

alloc_rov = decode_particle(pareto_front[best_rov_idx]['position'], n_fields, n_hosts, max_start_year, max_wells)
alloc_cvar = decode_particle(pareto_front[best_cvar_idx]['position'], n_fields, n_hosts, max_start_year, max_wells)

print(f'Best ROV:  E[ROV]={obj_final[best_rov_idx, 0]:.0f}, CVaR={obj_final[best_rov_idx, 1]:.0f} MNOK')
print(f'Best CVaR: E[ROV]={obj_final[best_cvar_idx, 0]:.0f}, CVaR={obj_final[best_cvar_idx, 1]:.0f} MNOK')

# Colors for plots
host_colors = ['steelblue', 'tomato', 'seagreen', 'darkorange', 'purple']
field_colors = plt.cm.tab20(np.linspace(0, 1, n_fields))

## 2. Field Summary

Each field has a unique combination of reservoir size, geological complexity, depth, and gas content.
These attributes drive both the production potential and the development cost.

In [ ]:
print(f'{"Field":<10} {"UR (MSm3)":>10} {"UR (mmbbl)":>11} {"Complexity":>11} {"Depth (m)":>10} '
      f'{"Wells":>6} {"GOR":>6} {"MaxRate":>8}')
print('-' * 74)
for i in range(n_fields):
    ur = field_data.ultimate_recovery[i]
    max_rate = field_data.n_wells[i] * per_well_rate * field_data.complexity_factor[i]
    print(f'{field_data.names[i]:<10} {ur:>10.1f} {ur * barrel:>11.0f} '
          f'{field_data.complexity_factor[i]:>11.2f} {field_data.depth[i]:>10.0f} '
          f'{field_data.n_wells[i]:>6d} {field_data.gor[i]:>6.0f} {max_rate:>8.2f}')

ur_all = field_data.ultimate_recovery * barrel
print(f'\nField size range: {ur_all.min():.0f} - {ur_all.max():.0f} mmbbl')
print(f'Depth range: {field_data.depth.min():.0f} - {field_data.depth.max():.0f} m')
print(f'Complexity range: {field_data.complexity_factor.min():.2f} - {field_data.complexity_factor.max():.2f}')

## 3. Host Summary

Host platforms have limited processing capacity for oil, gas, and water.
Operating costs vary significantly between hosts (2-20 USD/bbl variable cost).

In [ ]:
print(f'{"Host":<10} {"Oil Cap":>10} {"Gas Cap":>10} {"Water Cap":>10} '
      f'{"OPEX/y":>10} {"Var OPEX":>10} {"ABEX":>10}')
print(f'{"":<10} {"MSm3/y":>10} {"BSm3/y":>10} {"MSm3/y":>10} '
      f'{"MNOK":>10} {"USD/bbl":>10} {"MNOK":>10}')
print('-' * 72)
for j in range(n_hosts):
    print(f'{host_data.names[j]:<10} {host_data.oil_capacity[j].mean():>10.2f} '
          f'{host_data.gas_capacity[j].mean():>10.2f} {host_data.water_capacity[j].mean():>10.2f} '
          f'{cost_data.opex_rate[j]:>10.0f} {cost_data.variable_opex_rate[j]:>10.1f} '
          f'{cost_data.abex[j]:>10.0f}')

## 4. CAPEX Ranges

Development cost depends on three components: drilling (depth + well count), subsea infrastructure (fixed),
and pipeline (quadratic with distance). The table shows the cheapest host option per field.

In [ ]:
print(f'{"Field":<10} {"Drilling":>10} {"Subsea":>10} {"Pipeline":>10} {"Total":>10} '
      f'{"Total (bn)":>11} {"Best Host":>10}')
print(f'{"":<10} {"MNOK":>10} {"MNOK":>10} {"MNOK":>10} {"MNOK":>10} '
      f'{"NOK":>11} {"":>10}')
print('-' * 73)
for i in range(n_fields):
    j_best = capex_matrix[i].argmin()
    drilling = cost_data.well_cost_per_well[i] * field_data.n_wells[i]
    pipeline = cost_data.pipeline_cost[i, j_best]
    subsea = cost_data.subsea_base
    total = drilling + subsea + pipeline
    print(f'{field_data.names[i]:<10} {drilling:>10.0f} {subsea:>10.0f} {pipeline:>10.0f} '
          f'{total:>10.0f} {total/1000:>11.1f} {host_data.names[j_best]:>10}')

print(f'\nCAPEX range across all field-host pairs: '
      f'{capex_matrix.min()/1000:.1f} - {capex_matrix.max()/1000:.1f} bn NOK')

## 5. Spatial Map

Fields (circles) and hosts (squares) are placed in a 100 x 100 km area.
Pipeline cost increases quadratically with distance, so proximity to a host matters.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

for i in range(n_fields):
    for j in range(n_hosts):
        fx, fy = field_data.coords[i]
        hx, hy = host_data.coords[j]
        ax.plot([fx, hx], [fy, hy], 'k-', alpha=0.08, linewidth=0.5)

ax.scatter(field_data.coords[:, 0], field_data.coords[:, 1],
           s=120, c='steelblue', marker='o', zorder=5, edgecolors='white', linewidth=1.5)
for i, name in enumerate(field_data.names):
    ax.annotate(name, field_data.coords[i], fontsize=8, fontweight='bold',
                textcoords='offset points', xytext=(6, 6), color='steelblue')

ax.scatter(host_data.coords[:, 0], host_data.coords[:, 1],
           s=200, c='tomato', marker='s', zorder=5, edgecolors='white', linewidth=1.5)
for j, name in enumerate(host_data.names):
    ax.annotate(name, host_data.coords[j], fontsize=9, fontweight='bold',
                textcoords='offset points', xytext=(8, 8), color='tomato')

area = config['data_generation']['area_size_km']
ax.set_xlim(-10, area + 10)
ax.set_ylim(-10, area + 10)
ax.set_xlabel('x (km)')
ax.set_ylabel('y (km)')
ax.set_title('Field and Host Positions')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Production Profiles

Oil production per field at reference well count (before optimization).
Shape: ramp-up (1-2 years), plateau, then exponential decline.
Max rate = n_wells x per_well_rate x complexity_factor.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i in range(n_fields):
    axes[0].plot(years, field_data.oil[i], color=field_colors[i], linewidth=1.5, label=field_data.names[i])
    axes[1].plot(years, field_data.gas[i], color=field_colors[i], linewidth=1.5)
    axes[2].plot(years, field_data.water[i], color=field_colors[i], linewidth=1.5)

axes[0].set_ylabel('Oil (MSm3/y)')
axes[0].set_title('Oil Production')
axes[1].set_ylabel('Gas (BSm3/y)')
axes[1].set_title('Gas Production')
axes[2].set_ylabel('Water (MSm3/y)')
axes[2].set_title('Water Production')

for ax in axes:
    ax.set_xlabel('Year')
    ax.grid(True, alpha=0.3)

axes[0].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 7. Demand vs Capacity

Total field oil demand compared to available host capacity.
If demand exceeds capacity, the optimizer must choose which fields to develop and when.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

total_field_oil = field_data.oil.sum(axis=0)
ax.plot(years, total_field_oil, 'k--', linewidth=2, label='Total field demand')

for j in range(n_hosts):
    ax.plot(years, host_data.oil_capacity[j], color=host_colors[j],
            linewidth=1.5, label=f'{host_data.names[j]} capacity')

total_host_cap = host_data.oil_capacity.sum(axis=0)
ax.plot(years, total_host_cap, 'r-', linewidth=2, label='Total host capacity')

ax.set_xlabel('Year')
ax.set_ylabel('Oil (MSm3/y)')
ax.set_title('Host Capacity vs Field Demand (Oil)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

ratio = total_field_oil.max() / total_host_cap.mean()
print(f'Peak demand / avg capacity: {ratio:.1f}x')

---
# Part B: Optimization Results

The optimizer searches for portfolio configurations that maximize expected value (ROV)
while limiting downside risk (CVaR). The result is a set of trade-off solutions (Pareto front).

## 8. Pareto Front

Each point is a non-dominated portfolio. Top-right is ideal: high expected value with limited downside.
The trade-off curve shows how much expected return you must give up to reduce risk.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(obj_final[:, 1], obj_final[:, 0], c='seagreen', s=60, edgecolors='black', zorder=5)

ax.scatter(obj_final[best_rov_idx, 1], obj_final[best_rov_idx, 0],
           c='navy', s=120, edgecolors='black', zorder=6, marker='*')
ax.annotate(f'Best ROV\n({obj_final[best_rov_idx, 0]:.0f}, {obj_final[best_rov_idx, 1]:.0f})',
            (obj_final[best_rov_idx, 1], obj_final[best_rov_idx, 0]),
            textcoords='offset points', xytext=(10, -15), fontsize=9, fontweight='bold')

ax.scatter(obj_final[best_cvar_idx, 1], obj_final[best_cvar_idx, 0],
           c='darkred', s=120, edgecolors='black', zorder=6, marker='*')
ax.annotate(f'Best CVaR\n({obj_final[best_cvar_idx, 0]:.0f}, {obj_final[best_cvar_idx, 1]:.0f})',
            (obj_final[best_cvar_idx, 1], obj_final[best_cvar_idx, 0]),
            textcoords='offset points', xytext=(10, 10), fontsize=9, fontweight='bold')

ax.set_xlabel('CVaR 5% (MNOK)')
ax.set_ylabel('Expected ROV (MNOK)')
ax.set_title(f'Pareto Front ({len(pareto_front)} solutions, {config["price"]["n_scenarios_final"]} scenarios)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Solution Landscape

All Pareto-optimal portfolios sorted by expected ROV.
Shows which fields are developed, assigned to which host, at what timing and well count.

In [ ]:
print(f'Pareto front has {len(pareto_front)} solutions\n')
print(f'{"#":<4} {"ROV":>8} {"CVaR":>8}   Developed   Skipped')
print('-' * 80)

sorted_idx = np.argsort(-obj_final[:, 0])

for rank, i in enumerate(sorted_idx[:20]):
    alloc = decode_particle(pareto_front[i]['position'], n_fields, n_hosts, max_start_year, max_wells)
    obj = pareto_front[i]['objectives_final']

    developed_list = []
    skipped = []
    for f in range(n_fields):
        h = alloc.assignment[f]
        if h == -1:
            skipped.append(f'F{f}')
        else:
            developed_list.append(f'F{f}->H{h}@y{alloc.start_year[f]}({alloc.n_wells[f]}w)')

    dev_str = ', '.join(developed_list[:8])
    if len(developed_list) > 8:
        dev_str += f'... +{len(developed_list)-8}'
    skip_str = ', '.join(skipped) if skipped else 'none'

    print(f'{rank+1:<4} {obj[0]:>8.0f} {obj[1]:>8.0f}   {dev_str}')
    if skipped:
        print(f'{"":<22}Skipped: {skip_str}')

if len(pareto_front) > 20:
    print(f'\n... and {len(pareto_front) - 20} more solutions')

## 10. Tieback Map — Best ROV vs Best CVaR

Spatial view of the two extreme Pareto solutions.
Lines show field-to-host assignments; labels show start year and well count.

In [ ]:
def plot_tieback_map(ax, alloc, title):
    """Plot field/host positions with assigned tiebacks highlighted."""
    for i in range(n_fields):
        j = int(alloc.assignment[i])
        if j < 0:
            continue
        fx, fy = field_data.coords[i]
        hx, hy = host_data.coords[j]
        color = host_colors[j % len(host_colors)]
        ax.plot([fx, hx], [fy, hy], '-', color=color, linewidth=2.5, alpha=0.8)

    ax.scatter(field_data.coords[:, 0], field_data.coords[:, 1],
               s=100, c='steelblue', marker='o', zorder=5, edgecolors='white', linewidth=1.5)
    for i, name in enumerate(field_data.names):
        h = int(alloc.assignment[i])
        if h >= 0:
            label = f'{name} (yr{alloc.start_year[i]}, {alloc.n_wells[i]}w)'
        else:
            label = f'{name} (skip)'
        ax.annotate(label, field_data.coords[i], fontsize=7, fontweight='bold',
                    textcoords='offset points', xytext=(6, 6), color='steelblue')

    ax.scatter(host_data.coords[:, 0], host_data.coords[:, 1],
               s=200, c='tomato', marker='s', zorder=5, edgecolors='white', linewidth=1.5)
    for j, name in enumerate(host_data.names):
        ax.annotate(name, host_data.coords[j], fontsize=9, fontweight='bold',
                    textcoords='offset points', xytext=(8, 8), color='tomato')

    area = config['data_generation']['area_size_km']
    ax.set_xlim(-10, area + 10)
    ax.set_ylim(-10, area + 10)
    ax.set_xlabel('x (km)')
    ax.set_ylabel('y (km)')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

obj_rov = pareto_front[best_rov_idx]['objectives_final']
plot_tieback_map(axes[0], alloc_rov,
                 f'Best ROV: {obj_rov[0]:.0f} MNOK (CVaR {obj_rov[1]:.0f})')

obj_cvar = pareto_front[best_cvar_idx]['objectives_final']
plot_tieback_map(axes[1], alloc_cvar,
                 f'Best CVaR: {obj_cvar[1]:.0f} MNOK (ROV {obj_cvar[0]:.0f})')

plt.tight_layout()
plt.show()

## 11. Development Timeline

When does each field start producing? Colored by host assignment.
Bar length = active production period. Shows whether the optimizer staggers development.

In [ ]:
alloc = alloc_rov

# Compute capped oil profiles (same logic as evaluation.py)
oil_profiles_tl = np.zeros((n_fields, n_years))
max_rates_tl = np.zeros(n_fields)
for f in range(n_fields):
    if alloc.assignment[f] < 0:
        continue
    nw = int(alloc.n_wells[f])
    mr = nw * per_well_rate * field_data.complexity_factor[f]
    max_rates_tl[f] = mr
    oil_profiles_tl[f] = make_oil_profile(
        n_years, float(field_data.ultimate_recovery[f]),
        mr, int(field_data.ramp_years[f]), float(field_data.decline_rate[f]),
    )
t_tl = np.arange(n_years, dtype=float)
wc_tl = np.zeros((n_fields, n_years))
for f in range(n_fields):
    if alloc.assignment[f] < 0:
        continue
    wc_tl[f] = logistic_watercut(
        t_tl, float(field_data.wc_midpoint[f]), float(field_data.wc_initial[f]),
        float(field_data.wc_final[f]), float(field_data.wc_steepness[f]),
    )
capped_tl = compute_capped_production(
    oil_profiles_tl, host_data, alloc, n_years,
    max_rates=max_rates_tl, ramp_years=field_data.ramp_years,
    gor=field_data.gor, wc_fractions=wc_tl,
)
capped_oil_tl = np.zeros((n_fields, n_years))
for host_idx, hr in capped_tl.items():
    for li, fi in enumerate(hr['field_indices']):
        capped_oil_tl[fi] = hr['oil'][li]

# Group fields by host, sorted by start year
host_groups = {}
for f in range(n_fields):
    h = int(alloc.assignment[f])
    if h >= 0:
        host_groups.setdefault(h, []).append(f)
for h in host_groups:
    host_groups[h].sort(key=lambda f: alloc.start_year[f])

# Build row layout: host header rows + field rows
rows = []
y_labels = []
for h in sorted(host_groups):
    rows.append(('host', h, -1))
    y_labels.append(host_data.names[h])
    for f in host_groups[h]:
        rows.append(('field', h, f))
        y_labels.append(f'  {field_data.names[f]}')

n_rows = len(rows)
fig, ax = plt.subplots(figsize=(14, max(5, n_rows * 0.6)))

for row_i, (rtype, h, f) in enumerate(rows):
    if rtype == 'host':
        ax.axhspan(row_i - 0.5, row_i + 0.5,
                   color=host_colors[h % len(host_colors)], alpha=0.10, zorder=0)
        continue

    start = int(alloc.start_year[f])
    profile = capped_oil_tl[f]
    producing = np.where(profile > 1e-6)[0]

    color = host_colors[h % len(host_colors)]

    if len(producing) == 0:
        # Built but never produced (fully constrained out)
        ax.barh(row_i, 1, left=start, color='lightgray', edgecolor=color,
                linewidth=1, alpha=0.5, height=0.65)
        ax.plot(start, row_i, '|', color='black', markersize=10, markeredgewidth=2, zorder=5)
        continue

    first_prod = int(producing[0])
    last_prod = int(producing[-1])

    # Waiting period: built but no throughput yet (FCFS deferral)
    if first_prod > start:
        ax.barh(row_i, first_prod - start, left=start,
                color='white', edgecolor=color, linewidth=1.2,
                alpha=0.8, height=0.65, linestyle='--')

    # Active production period
    prod_duration = last_prod - first_prod + 1
    ax.barh(row_i, prod_duration, left=first_prod,
            color=color, edgecolor='white', linewidth=0.5, alpha=0.85, height=0.65)

    # Start year tick (when CAPEX is committed)
    ax.plot(start, row_i, '|', color='black', markersize=10, markeredgewidth=2, zorder=5)

    # Label inside production bar
    if prod_duration > 2:
        label = f'{alloc.n_wells[f]}w  |  {field_data.ultimate_recovery[f]*barrel:.0f} mmbbl'
        ax.text(first_prod + prod_duration / 2, row_i, label,
                ha='center', va='center', fontsize=7.5, fontweight='bold', color='white')

ax.set_yticks(range(n_rows))
ax.set_yticklabels(y_labels, fontsize=8.5)
ax.set_xlabel('Year')
ax.set_title('Development Timeline — Best ROV\n'
             'Tick = build year  |  dashed = waiting for capacity  |  solid = producing')
ax.set_xlim(-0.5, n_years + 0.5)
ax.set_ylim(-0.5, n_rows - 0.5)
ax.invert_yaxis()
ax.grid(True, alpha=0.25, axis='x')

used_hosts = sorted(host_groups.keys())
legend_elements = [Patch(facecolor=host_colors[h % len(host_colors)], edgecolor='white',
                         label=host_data.names[h]) for h in used_hosts]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()


## 12. Well Count Comparison

Reference well count (from data generation, based on UR and complexity) vs what the optimizer chose.
The optimizer may increase wells for profitable fields or decrease them for marginal ones.

In [ ]:
alloc = alloc_rov
ref_wells = field_data.n_wells
opt_wells = alloc.n_wells

x = np.arange(n_fields)
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, ref_wells, width, label='Reference (data gen)', color='lightgray', edgecolor='gray')
ax.bar(x + width/2, opt_wells, width, label='PSO optimized', color='steelblue', edgecolor='black')
ax.set_xlabel('Field')
ax.set_ylabel('Number of wells')
ax.set_title('Well Count: Reference vs PSO Optimized (Best ROV)')
ax.set_xticks(x)
ax.set_xticklabels(field_data.names, rotation=45, ha='right')
ax.axhline(max_wells, color='red', linestyle='--', alpha=0.5, label=f'Max wells ({max_wells})')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'Reference total wells: {ref_wells.sum()}')
print(f'Optimized total wells: {opt_wells.sum()}')
print(f'Fields at max wells: {(opt_wells >= max_wells).sum()}/{n_fields}')

## 13. Capped Production Profiles

Oil production after applying host capacity constraints.
Built from PSO-optimized well counts, shifted by start year, capped by host capacity.

In [ ]:
alloc = alloc_rov

# Build oil profiles from optimized n_wells
oil_profiles = np.zeros((n_fields, n_years))
max_rates = np.zeros(n_fields)

for f in range(n_fields):
    if alloc.assignment[f] < 0:
        continue
    n_wells_f = int(alloc.n_wells[f])
    max_rate = n_wells_f * per_well_rate * field_data.complexity_factor[f]
    max_rates[f] = max_rate
    oil_profiles[f] = make_oil_profile(
        n_years, float(field_data.ultimate_recovery[f]),
        max_rate, int(field_data.ramp_years[f]), float(field_data.decline_rate[f]),
    )

# Precompute water cut fractions at field-relative time
t = np.arange(n_years, dtype=float)
wc_fractions = np.zeros((n_fields, n_years))
for f in range(n_fields):
    if alloc.assignment[f] < 0:
        continue
    wc_fractions[f] = logistic_watercut(
        t, float(field_data.wc_midpoint[f]),
        float(field_data.wc_initial[f]),
        float(field_data.wc_final[f]),
        float(field_data.wc_steepness[f]),
    )

# Compute capped production (FCFS oil + derived gas/water)
capped = compute_capped_production(
    oil_profiles, host_data, alloc, n_years,
    max_rates=max_rates,
    ramp_years=field_data.ramp_years,
    gor=field_data.gor,
    wc_fractions=wc_fractions,
)

capped_oil_all = np.zeros((n_fields, n_years))
capped_gas_all = np.zeros((n_fields, n_years))
for host_idx, hr in capped.items():
    for li, fi in enumerate(hr['field_indices']):
        capped_oil_all[fi] = hr['oil'][li]
        capped_gas_all[fi] = hr['gas'][li]

fig, ax = plt.subplots(figsize=(14, 6))
developed = [f for f in range(n_fields) if alloc.assignment[f] >= 0]
developed_sorted = sorted(developed, key=lambda f: alloc.start_year[f])

bottom = np.zeros(n_years)
for f in developed_sorted:
    ax.fill_between(years, bottom, bottom + capped_oil_all[f],
                     alpha=0.7, color=field_colors[f], label=field_data.names[f])
    bottom += capped_oil_all[f]

ax.set_xlabel('Year')
ax.set_ylabel('Oil production (MSm3/y)')
ax.set_title('Capped Oil Production by Field (Best ROV)')
ax.legend(loc='upper right', fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

total_oil = capped_oil_all.sum()
print(f'Total portfolio oil production: {total_oil:.1f} MSm3 ({total_oil * barrel:.0f} mmbbl)')

## 14. Host Capacity Utilization

Oil demand vs processing capacity per host. Red shading = production lost to capacity constraints.
Shows whether the optimizer effectively uses available capacity.

In [ ]:
alloc = alloc_rov

# Build uncapped shifted profiles per host (original demand without deferral)
uncapped_per_host = {}
for host_idx in range(n_hosts):
    fields_on_host = [f for f in range(n_fields) if alloc.assignment[f] == host_idx]
    if not fields_on_host:
        continue
    uncapped_oil = np.zeros(n_years)
    for f in fields_on_host:
        start = int(alloc.start_year[f])
        profile = oil_profiles[f]
        usable = min(len(profile), n_years - start)
        if usable > 0:
            uncapped_oil[start:start + usable] += profile[:usable]
    uncapped_per_host[host_idx] = uncapped_oil

active_hosts = sorted(uncapped_per_host.keys())
n_active = len(active_hosts)

ncols = min(n_active, 3)
nrows = (n_active + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows), squeeze=False)

for plot_idx, host_idx in enumerate(active_hosts):
    row, col = divmod(plot_idx, ncols)
    ax = axes[row, col]
    capacity = host_data.oil_capacity[host_idx]
    demand = uncapped_per_host[host_idx]

    # Per-field capped production for this host
    fields_here = []
    field_capped = []
    if host_idx in capped:
        for li, fi in enumerate(capped[host_idx]['field_indices']):
            fields_here.append(fi)
            field_capped.append(capped[host_idx]['oil'][li])

    # Stacked area per field (shows which field produces when)
    bottom = np.zeros(n_years)
    for fi, fc in zip(fields_here, field_capped):
        ax.fill_between(years, bottom, bottom + fc,
                        alpha=0.7, color=field_colors[fi], label=field_data.names[fi])
        bottom += fc

    # Original demand (no deferral) and capacity
    ax.plot(years, demand, 'k--', linewidth=1.5, label='Original demand (no deferral)')
    ax.plot(years, capacity, 'r-', linewidth=2, label='Capacity')

    ax.set_title(f'{host_data.names[host_idx]} ({len(fields_here)} fields)')
    ax.set_xlabel('Year')
    ax.set_ylabel('Oil (MSm3/y)')
    ax.legend(fontsize=6, loc='upper right')
    ax.grid(True, alpha=0.3)

# Hide empty subplots
for idx in range(n_active, nrows * ncols):
    row, col = divmod(idx, ncols)
    axes[row, col].set_visible(False)

plt.suptitle('Host Capacity Utilization (Best ROV)', fontweight='bold')
plt.tight_layout()
plt.show()

## 14b. Capped Production per Host (Bar Chart)

Year-by-year stacked bar chart of capped oil production per host, colored by field.
Shows exactly when each field produces and how much, including deferred volume.

In [ ]:
alloc = alloc_rov

active_hosts = sorted(h for h in range(n_hosts)
                      if any(alloc.assignment[f] == h for f in range(n_fields)))
n_active = len(active_hosts)

ncols = min(n_active, 3)
nrows = (n_active + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)

for plot_idx, host_idx in enumerate(active_hosts):
    row, col = divmod(plot_idx, ncols)
    ax = axes[row, col]
    capacity = host_data.oil_capacity[host_idx]

    # Per-field capped production for this host
    fields_here = []
    field_capped = []
    if host_idx in capped:
        for li, fi in enumerate(capped[host_idx]['field_indices']):
            fields_here.append(fi)
            field_capped.append(capped[host_idx]['oil'][li])

    # Stacked bars per field
    bottom = np.zeros(n_years)
    for fi, fc in zip(fields_here, field_capped):
        ax.bar(years, fc, bottom=bottom, width=0.8,
               alpha=0.8, color=field_colors[fi], edgecolor='white',
               linewidth=0.3, label=field_data.names[fi])
        bottom += fc

    # Capacity line
    ax.plot(years, capacity, 'r-', linewidth=2, label='Capacity')

    ax.set_title(f'{host_data.names[host_idx]} ({len(fields_here)} fields)')
    ax.set_xlabel('Year')
    ax.set_ylabel('Oil (MSm3/y)')
    ax.legend(fontsize=6, loc='upper right')
    ax.grid(True, alpha=0.3, axis='y')

# Hide empty subplots
for idx in range(n_active, nrows * ncols):
    row, col = divmod(idx, ncols)
    axes[row, col].set_visible(False)

plt.suptitle('Capped Production per Host — Bar Chart (Best ROV)', fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part C: Economics & Risk

Detailed economic analysis of the best ROV portfolio.
All probabilistic outputs come from oil/gas price scenario variation — production is deterministic given an allocation.

## 15. Reserves per Host

Total recoverable oil from capped profiles, grouped by host platform.

In [ ]:
alloc = alloc_rov
developed = [f for f in range(n_fields) if alloc.assignment[f] >= 0]
field_oil_reserves = capped_oil_all.sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
active_hosts = sorted(set(int(alloc.assignment[f]) for f in developed))

x = np.arange(len(active_hosts))
bottom = np.zeros(len(active_hosts))

for f in developed:
    h = int(alloc.assignment[f])
    h_pos = active_hosts.index(h)
    reserves_mmbbl = field_oil_reserves[f] * barrel
    ax.bar(h_pos, reserves_mmbbl, bottom=bottom[h_pos],
           color=field_colors[f], edgecolor='black', label=field_data.names[f])
    if reserves_mmbbl > 1:
        ax.text(h_pos, bottom[h_pos] + reserves_mmbbl / 2,
                f'{reserves_mmbbl:.0f}', ha='center', va='center', fontsize=7)
    bottom[h_pos] += reserves_mmbbl

for i, h in enumerate(active_hosts):
    ax.text(i, bottom[i] + 1, f'{bottom[i]:.0f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([host_data.names[h] for h in active_hosts])
ax.set_ylabel('Recoverable Oil (mmbbl)')
ax.set_title('Reserves per Host (capped)')
ax.legend(loc='upper right', fontsize=7, ncol=2)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

total_mmbbl = field_oil_reserves.sum() * barrel
print(f'Total portfolio reserves: {field_oil_reserves.sum():.1f} MSm3 ({total_mmbbl:.0f} mmbbl)')

## 16. CAPEX Breakdown & Timeline

Left: per-field CAPEX split into drilling, subsea, and pipeline components.
Right: when the investment hits (CAPEX at start year of each field).

In [ ]:
alloc = alloc_rov
developed = [f for f in range(n_fields) if alloc.assignment[f] >= 0]

field_capex = {}
drilling_costs = []
subsea_costs = []
pipeline_costs = []
labels = []

for f in developed:
    h = int(alloc.assignment[f])
    n_w = int(alloc.n_wells[f])
    drill = float(cost_data.well_cost_per_well[f]) * n_w
    subsea = cost_data.subsea_base
    pipe = float(cost_data.pipeline_cost[f, h])
    field_capex[f] = drill + subsea + pipe
    drilling_costs.append(drill)
    subsea_costs.append(subsea)
    pipeline_costs.append(pipe)
    labels.append(f'{field_data.names[f]}\n({n_w}w)')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ax = axes[0]
x = np.arange(len(developed))
ax.bar(x, drilling_costs, label='Drilling', color='steelblue', edgecolor='black')
ax.bar(x, subsea_costs, bottom=drilling_costs, label='Subsea', color='seagreen', edgecolor='black')
bottom2 = np.array(drilling_costs) + np.array(subsea_costs)
ax.bar(x, pipeline_costs, bottom=bottom2, label='Pipeline', color='sandybrown', edgecolor='black')

totals = np.array(drilling_costs) + np.array(subsea_costs) + np.array(pipeline_costs)
for i, total in enumerate(totals):
    ax.text(i, total + 50, f'{total/1000:.1f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel('CAPEX (MNOK)')
ax.set_title('CAPEX Breakdown (bn NOK labels)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
capex_per_year = np.zeros(n_years)
for f in developed:
    start = int(alloc.start_year[f])
    capex_per_year[start] += field_capex[f]

bottom_timeline = np.zeros(n_years)
for f in sorted(developed, key=lambda f: alloc.start_year[f]):
    start = int(alloc.start_year[f])
    capex_arr = np.zeros(n_years)
    capex_arr[start] = field_capex[f]
    ax.bar(years, capex_arr, bottom=bottom_timeline, color=field_colors[f],
           edgecolor='black', label=f'{field_data.names[f]} ({field_capex[f]/1000:.1f} bn)')
    bottom_timeline += capex_arr

ax.set_xlabel('Year')
ax.set_ylabel('CAPEX (MNOK)')
ax.set_title('CAPEX Timeline')
ax.legend(loc='upper right', fontsize=7, ncol=2)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f'Total portfolio CAPEX: {sum(field_capex.values()):.0f} MNOK ({sum(field_capex.values())/1000:.1f} bn NOK)')
print(f'  Drilling: {sum(drilling_costs)/1000:.1f} bn ({sum(drilling_costs)/sum(field_capex.values())*100:.0f}%)')
print(f'  Subsea:   {sum(subsea_costs)/1000:.1f} bn ({sum(subsea_costs)/sum(field_capex.values())*100:.0f}%)')
print(f'  Pipeline: {sum(pipeline_costs)/1000:.1f} bn ({sum(pipeline_costs)/sum(field_capex.values())*100:.0f}%)')

## 17. Cash Flow

Annual and cumulative portfolio cash flow. Revenue varies with oil/gas prices (P10/P50/P90 fan chart).
CAPEX, OPEX, and ABEX are deterministic.

In [ ]:
alloc = alloc_rov
prices = generate_price_scenarios_final(config)

portfolio_oil = capped_oil_all.sum(axis=0)
portfolio_gas = capped_gas_all.sum(axis=0)

oil_revenue = prices.oil * (portfolio_oil[np.newaxis, :] * barrel * nok_per_usd)
gas_revenue = prices.gas * (portfolio_gas[np.newaxis, :] * btu * nok_per_usd)
revenue = oil_revenue + gas_revenue

cost_per_year = np.zeros(n_years)
developed = [f for f in range(n_fields) if alloc.assignment[f] >= 0]

for f in developed:
    h = int(alloc.assignment[f])
    n_on_host = sum(1 for ff in developed if alloc.assignment[ff] == h)
    shared_opex = float(cost_data.opex_rate[h]) / n_on_host
    var_opex = float(cost_data.variable_opex_rate[h]) * capped_oil_all[f] * barrel * nok_per_usd
    field_active = capped_oil_all[f] > 0
    cost_per_year += shared_opex * field_active + var_opex
    cost_per_year[int(alloc.start_year[f])] += field_capex[f]
    if field_active.any():
        last_year = int(np.max(np.where(field_active)))
        cost_per_year[last_year] += float(cost_data.abex[h])

net_cashflow = revenue - cost_per_year[np.newaxis, :]
cum_cashflow = np.cumsum(net_cashflow, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ax = axes[0]
p10 = np.percentile(net_cashflow, 10, axis=0)
p50 = np.percentile(net_cashflow, 50, axis=0)
p90 = np.percentile(net_cashflow, 90, axis=0)
ax.fill_between(years, p10, p90, alpha=0.2, color='steelblue', label='P10-P90')
ax.plot(years, p50, '-', color='steelblue', linewidth=2, label='P50 (median)')
ax.plot(years, np.mean(net_cashflow, axis=0), '--', color='navy', linewidth=1.5, label='Mean')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Net Cash Flow (MNOK)')
ax.set_title('Annual Portfolio Cash Flow (undiscounted)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
cp10 = np.percentile(cum_cashflow, 10, axis=0)
cp50 = np.percentile(cum_cashflow, 50, axis=0)
cp90 = np.percentile(cum_cashflow, 90, axis=0)
ax.fill_between(years, cp10, cp90, alpha=0.2, color='seagreen', label='P10-P90')
ax.plot(years, cp50, '-', color='seagreen', linewidth=2, label='P50 (median)')
ax.plot(years, np.mean(cum_cashflow, axis=0), '--', color='darkgreen', linewidth=1.5, label='Mean')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Cumulative Cash Flow (MNOK)')
ax.set_title('Cumulative Portfolio Cash Flow (undiscounted)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

payback_idx = np.where(cp50 > 0)[0]
if len(payback_idx) > 0:
    print(f'Median payback period: year {payback_idx[0]}')
else:
    print('Median scenario does not reach payback within planning horizon')

## 18. ROV Distribution

Distribution of portfolio Real Option Value across price scenarios.
The red-shaded tail (worst 5%) represents the CVaR — the expected loss in the worst outcomes.

In [ ]:
result_rov = evaluate_portfolio(alloc_rov, field_data, host_data, cost_data, prices, config)
scenario_values = result_rov.scenario_values

alpha = val_cfg['cvar_alpha']
var_threshold = np.percentile(scenario_values, alpha * 100)

fig, ax = plt.subplots(figsize=(12, 5))

n_bins = 60
counts, bins, patches = ax.hist(scenario_values, bins=n_bins, color='steelblue',
                                 edgecolor='black', alpha=0.7)

for patch, left_edge in zip(patches, bins[:-1]):
    if left_edge < var_threshold:
        patch.set_facecolor('tomato')
        patch.set_alpha(0.8)

ax.axvline(result_rov.expected_rov, color='navy', linestyle='--', linewidth=2,
           label=f'Mean: {result_rov.expected_rov:.0f} MNOK')
ax.axvline(var_threshold, color='red', linestyle=':', linewidth=1.5,
           label=f'VaR {alpha:.0%}: {var_threshold:.0f} MNOK')
ax.axvline(np.percentile(scenario_values, 10), color='gray', linestyle=':', linewidth=1,
           label=f'P10: {np.percentile(scenario_values, 10):.0f} MNOK')
ax.axvline(np.percentile(scenario_values, 90), color='gray', linestyle='--', linewidth=1,
           label=f'P90: {np.percentile(scenario_values, 90):.0f} MNOK')

ax.set_xlabel('Portfolio ROV (MNOK)')
ax.set_ylabel('Count')
ax.set_title('Portfolio ROV Distribution')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'E[ROV]: {result_rov.expected_rov:.0f} MNOK')
print(f'CVaR {alpha:.0%}: {result_rov.cvar:.0f} MNOK')
print(f'Std: {scenario_values.std():.0f} MNOK')
print(f'P(ROV < 0): {(scenario_values < 0).mean():.1%}')

## 19. Exceedance Curve & Breakeven Price

Left: probability of exceeding a given portfolio ROV threshold.
Right: expected portfolio NPV as a function of oil price level. The breakeven is where NPV crosses zero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
sorted_vals = np.sort(scenario_values)
exceedance_prob = 1.0 - np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)

ax.plot(sorted_vals, exceedance_prob, '-', color='steelblue', linewidth=2)
ax.axvline(0, color='red', linestyle=':', alpha=0.7,
           label=f'ROV=0: P={exceedance_prob[np.searchsorted(sorted_vals, 0)]:.1%}')
ax.axvline(result_rov.expected_rov, color='navy', linestyle='--', alpha=0.7,
           label=f'Mean: P={exceedance_prob[np.searchsorted(sorted_vals, result_rov.expected_rov)]:.1%}')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Threshold (MNOK)')
ax.set_ylabel('P(ROV > threshold)')
ax.set_title('Exceedance Probability Curve')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.02, 1.02)

ax = axes[1]
multipliers = np.linspace(0.2, 1.8, 20)
expected_npvs = []

for k in multipliers:
    scaled_oil = prices.oil * k
    portfolio_npv = np.zeros(prices.n_scenarios)

    for host_idx, hr in capped.items():
        n_on_host = len(hr['field_indices'])
        for li, fi in enumerate(hr['field_indices']):
            start = int(alloc_rov.start_year[fi])
            capex = field_capex[fi]
            shared_opex = float(cost_data.opex_rate[host_idx]) / n_on_host

            npvs = compute_field_npvs(
                capped_oil=hr['oil'][li], capped_gas=hr['gas'][li],
                oil_prices=scaled_oil, gas_prices=prices.gas,
                capex=capex, opex_rate=shared_opex,
                variable_opex_rate=float(cost_data.variable_opex_rate[host_idx]),
                abex=float(cost_data.abex[host_idx]),
                discount_rate=val_cfg['discount_rate'], start_year=start,
                nok_per_usd=nok_per_usd, barrel=barrel, btu=btu,
            )
            static_npv = npvs[:, start] * np.exp(-val_cfg['discount_rate'] * start)
            static_npv = np.maximum(static_npv, 0)
            portfolio_npv += static_npv

    expected_npvs.append(np.mean(portfolio_npv))

expected_npvs = np.array(expected_npvs)
base_price = np.exp(config['price']['oil']['E0'])

if expected_npvs[0] < 0 and expected_npvs[-1] > 0:
    breakeven_k = np.interp(0, expected_npvs, multipliers)
else:
    breakeven_k = None

ax.plot(multipliers * base_price, expected_npvs, 'o-', color='steelblue', linewidth=2, markersize=4)
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(base_price, color='gray', linestyle='--', alpha=0.5, label=f'Base price ({base_price:.0f} USD/bbl)')

if breakeven_k is not None:
    be_price = breakeven_k * base_price
    ax.axvline(be_price, color='red', linestyle=':', linewidth=1.5,
               label=f'Breakeven: {be_price:.0f} USD/bbl ({breakeven_k:.2f}x)')
    ax.plot(be_price, 0, 'ro', markersize=8, zorder=5)

ax.set_xlabel('Oil Price Level (USD/bbl)')
ax.set_ylabel('Expected Portfolio NPV (MNOK)')
ax.set_title('Breakeven Oil Price (static NPV, no optionality)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

if breakeven_k is not None:
    print(f'Breakeven oil price: {be_price:.0f} USD/bbl ({breakeven_k:.2f}x base)')
else:
    print('No breakeven found in range \u2014 portfolio profitable across entire range')
    print(f'NPV at lowest price ({multipliers[0]*base_price:.0f} USD/bbl): {expected_npvs[0]:.0f} MNOK')

## 20. Field-Level ROV Contribution

Per-field expected ROV for the best ROV portfolio.
Negative values mean the field destroys portfolio value (development cost exceeds expected revenue).

In [ ]:
field_rovs = result_rov.field_values

fig, ax = plt.subplots(figsize=(10, 7))

colors = ['steelblue' if v >= 0 else 'tomato' for v in field_rovs]
y = np.arange(n_fields)
ax.barh(y, field_rovs, color=colors, edgecolor='black')
ax.set_yticks(y)
ax.set_yticklabels([f'{field_data.names[f]} (H{alloc_rov.assignment[f]}, {alloc_rov.n_wells[f]}w)'
                     if alloc_rov.assignment[f] >= 0
                     else f'{field_data.names[f]} (skip)'
                     for f in range(n_fields)], fontsize=9)
ax.set_xlabel('Expected ROV (MNOK)')
ax.set_title('Per-Field ROV Contribution (Best ROV Solution)')
ax.axvline(0, color='black', linewidth=0.8)
ax.grid(True, alpha=0.3, axis='x')

for i, v in enumerate(field_rovs):
    offset = max(abs(field_rovs).max() * 0.02, 50)
    ax.text(v + (offset if v >= 0 else -offset), i, f'{v:.0f}', va='center',
            ha='left' if v >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.show()

print(f'Portfolio E[ROV]: {result_rov.expected_rov:.0f} MNOK')
print(f'Sum of field ROVs: {field_rovs.sum():.0f} MNOK')
n_positive = (field_rovs > 0).sum()
n_negative = (field_rovs < 0).sum()
n_skip = (np.array([alloc_rov.assignment[f] for f in range(n_fields)]) < 0).sum()
print(f'Fields: {n_positive} positive, {n_negative} negative, {n_skip} skipped')